In [1]:
# NLP + K-MEANS CLUSTERING

import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

nltk.download("stopwords")

# 1. Load dataset
df = pd.read_csv("merged.csv")

# 2. Select text columns
text_cols = df.select_dtypes(include="object").columns
df["text"] = df[text_cols].fillna("").astype(str).agg(" ".join, axis=1)

# 3. NLP cleaning
stop_words = set(stopwords.words("english"))

def clean(x):
    x = re.sub(r"[^a-zA-Z\s]", "", x.lower())
    return " ".join(w for w in x.split() if w not in stop_words)

df["clean_text"] = df["text"].apply(clean)

# 4. TF-IDF
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df["clean_text"])

# 5. Find best K using silhouette score
scores = {}

for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    scores[k] = silhouette_score(X, labels)

best_k = max(scores, key=scores.get)
print("Best K:", best_k)
print("Silhouette Score:", scores[best_k])

# 6. K-Means
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

# 7. Show cluster sizes
print("\nCluster Sizes:")
print(df["cluster"].value_counts().sort_index())

# 8. Show top words in each cluster
words = tfidf.get_feature_names_out()

for i in range(best_k):
    top = kmeans.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:")
    print(", ".join(words[j] for j in top))

# 9. Save results
df.drop(columns=["text", "clean_text"]).to_csv(
    "merged_kmeans.csv", index=False
)

print("\nSaved as merged_kmeans.csv")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_5748/4127008595.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns


Best K: 2
Silhouette Score: 0.2990870378613799

Cluster Sizes:
cluster
0    292
1     90
Name: count, dtype: int64

Cluster 0:
yes, mother, gp, gt, services, course, le, teacher, home, reputation

Cluster 1:
yes, father, services, gt, gp, home, reputation, course, le, health

Saved as merged_kmeans.csv
